dim_nighbourhood


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS airbnb_barcelona_project.gold")

listings_silver = spark.table("airbnb_barcelona_project.silver.listings")

dim_neighbourhood = listings_silver.select(
    "neighbourhood_cleansed", "neighbourhood_group_cleansed"
).distinct()

dim_neighbourhood.write.format("delta").mode("overwrite") \
    .saveAsTable("airbnb_barcelona_project.gold.dim_neighbourhood")

print(f"dim_neighbourhood: {dim_neighbourhood.count()} rows")

dim_neighbourhood: 69 rows


In [0]:
dim_host = listings_silver.select(
    "host_id", "host_name", "host_since", "host_is_superhost",
    "host_response_rate", "host_acceptance_rate"
).distinct()

dim_host.write.format("delta").mode("overwrite") \
    .saveAsTable("airbnb_barcelona_project.gold.dim_host")

print(f"dim_host: {dim_host.count()} rows")

dim_host: 4595 rows


In [0]:
fact_listings = listings_silver.select(
    "id", "host_id", "neighbourhood_cleansed", "property_type", "room_type",
    "accommodates", "bedrooms", "beds", "price", "minimum_nights",
    "availability_30", "availability_365", "number_of_reviews",
    "review_scores_rating", "review_scores_cleanliness", "review_scores_location",
    "review_scores_value", "reviews_per_month", "estimated_occupancy_l365d",
    "estimated_revenue_l365d", "license_status", "instant_bookable"
)

fact_listings.write.format("delta").mode("overwrite") \
    .saveAsTable("airbnb_barcelona_project.gold.fact_listings")

print(f"fact_listings: {fact_listings.count()} rows")

fact_listings: 15293 rows


In [0]:
from pyspark.sql.functions import col, sum as _sum, count as _count, when

calendar_silver = spark.table("airbnb_barcelona_project.silver.calendar")

daily_occupancy = (calendar_silver
    .groupBy("date")
    .agg(
        _count("*").alias("total_listings"),
        _sum(when(col("available") == False, 1).otherwise(0)).alias("booked_count")
    )
    .withColumn("occupancy_rate", col("booked_count") / col("total_listings"))
)

daily_occupancy.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airbnb_barcelona_project.gold.daily_occupancy")

print(f"daily_occupancy: {daily_occupancy.count()} rows")
display(daily_occupancy.orderBy("date").limit(10))

daily_occupancy: 374 rows


date,total_listings,booked_count,occupancy_rate
2026-06-24,2341,2081,0.8889363519863306
2026-06-25,13035,10496,0.805216724204066
2026-06-26,13035,10136,0.7775987725354814
2026-06-27,13035,9980,0.7656309934790947
2026-06-28,13035,8993,0.6899117759877254
2026-06-29,13035,8652,0.6637514384349827
2026-06-30,13035,8347,0.6403528960490986
2026-07-01,13035,8422,0.6461066359800537
2026-07-02,13560,9348,0.6893805309734513
2026-07-03,15406,11819,0.7671686355965208


In [0]:
fact_listings = spark.table("airbnb_barcelona_project.gold.fact_listings")

fact_listings_with_geo = listings_silver.select(
    "id", "host_id", "neighbourhood_cleansed", "property_type", "room_type",
    "accommodates", "bedrooms", "beds", "price", "minimum_nights",
    "availability_30", "availability_365", "number_of_reviews",
    "review_scores_rating", "review_scores_cleanliness", "review_scores_location",
    "review_scores_value", "reviews_per_month", "estimated_occupancy_l365d",
    "estimated_revenue_l365d", "license_status", "instant_bookable",
    "latitude", "longitude"
)

fact_listings_with_geo.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airbnb_barcelona_project.gold.fact_listings")

print(f"fact_listings (with geo): {fact_listings_with_geo.count()} rows")

fact_listings (with geo): 15293 rows


In [0]:
from pyspark.sql.functions import col, avg
fact = spark.table("airbnb_barcelona_project.gold.fact_listings")
print(fact.filter(col("price_outlier") == False).agg(avg("price")).collect())

[Row(avg(price)=219.79202705766758)]


In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

dh = spark.table("airbnb_barcelona_project.gold.dim_host")
w = Window.partitionBy("host_id").orderBy(col("host_since"))
dh_clean = dh.withColumn("rn", row_number().over(w)).filter("rn = 1").drop("rn")

dh_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airbnb_barcelona_project.gold.dim_host")

print(dh_clean.count(), dh_clean.select("host_id").distinct().count())

4595 4595


In [0]:
from pyspark.sql.functions import col, avg
f = spark.table("airbnb_barcelona_project.gold.fact_listings")
print(f.filter(col("price_outlier") == False).agg(avg("price")).collect())

[Row(avg(price)=219.79202705766758)]


In [0]:
from pyspark.sql.functions import col, sum as _sum
f = spark.table("airbnb_barcelona_project.gold.fact_listings")
(f.filter(col("price_outlier") == False)
  .groupBy("neighbourhood_cleansed")
  .agg(_sum("estimated_revenue_l365d").alias("est_rev"))
  .orderBy(col("est_rev").desc())
  .show(10, False))

+-------------------------------------+--------+
|neighbourhood_cleansed               |est_rev |
+-------------------------------------+--------+
|la Dreta de l'Eixample               |60529473|
|la Sagrada Família                   |25926728|
|la Vila de Gràcia                    |20650924|
|l'Antiga Esquerra de l'Eixample      |19876874|
|Sant Antoni                          |18789435|
|el Poble Sec                         |15011507|
|el Barri Gòtic                       |11888154|
|el Fort Pienc                        |11686019|
|Sant Pere, Santa Caterina i la Ribera|11397746|
|la Nova Esquerra de l'Eixample       |11295672|
+-------------------------------------+--------+
only showing top 10 rows


In [0]:
from pyspark.sql.functions import col, count, sum as _sum, when, round as _round
f = spark.table("airbnb_barcelona_project.gold.fact_listings")
(f.groupBy("neighbourhood_cleansed")
  .agg(count("*").alias("listings"),
       _round(_sum(when(col("license_status") == "No License Shown", 1).otherwise(0)) / count("*") * 100, 1).alias("no_license_pct"))
  .filter(col("listings") >= 100)
  .orderBy(col("no_license_pct").desc())
  .show(10, False))

+--------------------------------------------+--------+--------------+
|neighbourhood_cleansed                      |listings|no_license_pct|
+--------------------------------------------+--------+--------------+
|el Besòs i el Maresme                       |100     |45.0          |
|el Guinardó                                 |102     |31.4          |
|el Raval                                    |1031    |27.2          |
|Sants - Badal                               |149     |26.8          |
|la Barceloneta                              |410     |26.1          |
|el Parc i la Llacuna del Poblenou           |161     |24.2          |
|Diagonal Mar i el Front Marítim del Poblenou|119     |23.5          |
|el Barri Gòtic                              |881     |23.4          |
|el Camp d'en Grassot i Gràcia Nova          |257     |23.3          |
|la Nova Esquerra de l'Eixample              |660     |22.4          |
+--------------------------------------------+--------+--------------+
only s

In [0]:
f = spark.table("airbnb_barcelona_project.gold.fact_listings")
f.groupBy("room_type", "license_status").count().orderBy("room_type", "license_status").show(30, False)

+---------------+----------------+-----+
|room_type      |license_status  |count|
+---------------+----------------+-----+
|Entire home/apt|Exempt          |2318 |
|Entire home/apt|Licensed        |7380 |
|Entire home/apt|No License Shown|1139 |
|Entire home/apt|Other/Unclear   |7    |
|Hotel room     |Exempt          |32   |
|Hotel room     |Licensed        |33   |
|Hotel room     |Other/Unclear   |3    |
|Private room   |Exempt          |1486 |
|Private room   |Licensed        |855  |
|Private room   |No License Shown|1895 |
|Private room   |Other/Unclear   |26   |
|Shared room    |Exempt          |88   |
|Shared room    |Licensed        |20   |
|Shared room    |No License Shown|9    |
|Shared room    |Other/Unclear   |2    |
+---------------+----------------+-----+



In [0]:
spark.table("airbnb_barcelona_project.gold.daily_occupancy") \
    .selectExpr("min(date)", "max(date)", "round(avg(occupancy_rate),3)").show()

+----------+----------+-----------------------------+
| min(date)| max(date)|round(avg(occupancy_rate), 3)|
+----------+----------+-----------------------------+
|2026-06-24|2027-07-02|                        0.426|
+----------+----------+-----------------------------+



In [0]:
from pyspark.sql.functions import col, sum as _sum, avg
d = spark.table("airbnb_barcelona_project.gold.daily_occupancy")
d.agg((_sum("booked_count") / _sum("total_listings")).alias("weighted"), avg("occupancy_rate").alias("simple")).show()
d.orderBy("date").select("date", "total_listings", "occupancy_rate").show(12)
d.orderBy(col("date").desc()).select("date", "total_listings", "occupancy_rate").show(12)

+------------------+------------------+
|          weighted|            simple|
+------------------+------------------+
|0.4165166035648804|0.4259585272439665|
+------------------+------------------+

+----------+--------------+------------------+
|      date|total_listings|    occupancy_rate|
+----------+--------------+------------------+
|2026-06-24|          2341|0.8889363519863306|
|2026-06-25|         13035| 0.805216724204066|
|2026-06-26|         13035|0.7775987725354814|
|2026-06-27|         13035|0.7656309934790947|
|2026-06-28|         13035|0.6899117759877254|
|2026-06-29|         13035|0.6637514384349827|
|2026-06-30|         13035|0.6403528960490986|
|2026-07-01|         13035|0.6461066359800537|
|2026-07-02|         13560|0.6893805309734513|
|2026-07-03|         15406|0.7671686355965208|
|2026-07-04|         15406|0.7639880566013242|
|2026-07-05|         15406|0.7021290406335194|
+----------+--------------+------------------+
only showing top 12 rows
+----------+----------

In [0]:
spark.table("airbnb_barcelona_project.bronze.listings").groupBy("calendar_last_scraped").count().orderBy("calendar_last_scraped").show(20)

+---------------------+-----+
|calendar_last_scraped|count|
+---------------------+-----+
|           2026-06-24| 2340|
|           2026-06-25|10695|
|           2026-07-02|  520|
|           2026-07-03| 1738|
+---------------------+-----+



In [0]:
from pyspark.sql.functions import col, max as _max
d = spark.table("airbnb_barcelona_project.gold.daily_occupancy")
mx = d.agg(_max("total_listings")).first()[0]
d.filter(col("total_listings") == mx).selectExpr("min(date)", "max(date)", "count(*)").show()

+----------+----------+--------+
| min(date)| max(date)|count(1)|
+----------+----------+--------+
|2026-07-03|2027-06-23|     356|
+----------+----------+--------+

